In [1]:
using Pkg
Pkg.status()

Status `~/Documents/rotation_project/Project.toml`
  [9e226e20] SpeedyWeather v0.18.1


In [2]:
using SpeedyWeather
spectral_grid = SpectralGrid()

SpectralGrid{Spectrum{...}, OctahedralGaussianGrid{...}}
├ Number format: Float32
├ Spectral:      T31 LowerTriangularMatrix
├ Grid:          48-ring OctahedralGaussianGrid, 3168 grid points
├ Resolution:    3.61°, 401km (at 6371km radius)
├ Vertical:      8-layer atmosphere
└ Architecture:  CPU using Array

In [3]:
# at = 80
flux = 0.1 * 1365
outdir_at = "/Users/woodh/Documents/rotation_project/stability"
fname = joinpath(outdir_at, "output.nc") 
output = NetCDFOutput(spectral_grid; filename=fname)

NetCDFOutput{Field{Float32, 1, Vector{Float32}, FullGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ status: inactive/uninitialized
├ write restart file: true (if active)
├ interpolator: AnvilInterpolator{Float32, RingGrids.GridGeometry{OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Vector{Float32}, Vector{Int64}, Vector{UnitRange{Int64}}}, RingGrids.AnvilLocator{Vector{Float32}, Vector{Int64}}}
├ path: /Users/woodh/Documents/rotation_project/stability/output.nc (overwrite=false)
├ frequency: 21600 seconds
└┐ variables:
 ├ v: meridional wind [m/s]
 ├ u: zonal wind [m/s]
 └ vor: relative vorticity [s^-1]

In [4]:
add!(output, SpeedyWeather.DivergenceOutput()) 
add!(output, SpeedyWeather.VorticityOutput())
add!(output, SpeedyWeather.TemperatureOutput())
add!(output, SpeedyWeather.SurfaceShortwaveDownOutput())
add!(output, SpeedyWeather.SoilMoistureOutput())
add!(output, SpeedyWeather.SoilTemperatureOutput())
add!(output, SpeedyWeather.SurfaceTemperatureOutput())
add!(output, SpeedyWeather.LandSeaMaskOutput())

NetCDFOutput{Field{Float32, 1, Vector{Float32}, FullGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ status: inactive/uninitialized
├ write restart file: true (if active)
├ interpolator: AnvilInterpolator{Float32, RingGrids.GridGeometry{OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Vector{Float32}, Vector{Int64}, Vector{UnitRange{Int64}}}, RingGrids.AnvilLocator{Vector{Float32}, Vector{Int64}}}
├ path: /Users/woodh/Documents/rotation_project/stability/output.nc (overwrite=false)
├ frequency: 21600 seconds
└┐ variables:
 ├ lsm: land-sea mask (1=land, 0=sea) [1]
 ├ st: soil temperature [degC]
 ├ tsurf: Surface air temperature [degC]
 ├ sm: soil moisture [1]
 ├ v: meridional wind [m/s]
 ├ temp: temperature [degC]
 ├ div: divergence [s^-1]
 ├ u: zonal wind [m/s]
 ├ srd: Surface shortwave radiation down [W/m^2]
 └ vor: relative vorticity [s^-1]

In [5]:
# p = Earth(spectral_grid, axial_tilt=at)
p = Earth(spectral_grid, solar_constant=flux)

Earth{Float32} <: SpeedyWeather.AbstractPlanet
├ radius::Float32 = 6.371e6
├ rotation::Float32 = 7.29e-5
├ gravity::Float32 = 9.81
├ daily_cycle::Bool = true
├ length_of_day::Second = 86400 seconds
├ seasonal_cycle::Bool = true
├ length_of_year::Second = 31557600 seconds
├ equinox::DateTime = 2000-03-20T00:00:00
├ axial_tilt::Float32 = 23.4
└ solar_constant::Float32 = 136.5

In [6]:
# radiative_transfer = OneBandShortwaveRadiativeTransfer(spectral_grid, ozone_absorption=0.0001)
radiative_transfer = OneBandShortwaveRadiativeTransfer(spectral_grid)

OneBandShortwaveRadiativeTransfer{Float32, SpeedyWeather.var"#OneBandShortwaveRadiativeTransfer##0#OneBandShortwaveRadiativeTransfer##1"} <: SpeedyWeather.AbstractShortwaveRadiativeTransfer
├ ozone_absorption::Float32 = 0.01
└ ozone_distribution::SpeedyWeather.var"#OneBandShortwaveRadiativeTransfer##0#OneBandShortwaveRadiativeTransfer##1" = #OneBandShortwaveRadiativeTransfer##0

In [7]:
shortwave_radiation = OneBandShortwave(spectral_grid; radiative_transfer)

OneBandShortwave <: AbstractShortwave
├ clouds: DiagnosticClouds{Float32}
├ transmissivity: BackgroundShortwaveTransmissivity{Float32}
└ radiative_transfer: OneBandShortwaveRadiativeTransfer{Float32, SpeedyWeather.var"#OneBandShortwaveRadiativeTransfer##0#OneBandShortwaveRadiativeTransfer##1"}

In [8]:
model = PrimitiveWetModel(spectral_grid; shortwave_radiation, output=output, planet=p)

PrimitiveWetModel <: PrimitiveWet
├ spectral_grid: SpectralGrid{CPU{KernelAbstractions.CPU}, Spectrum{CPU{KernelAbstractions.CPU}...
├ architecture: CPU{KernelAbstractions.CPU}
├ dynamics: Bool
├ geometry: Geometry{SpectralGrid{CPU{KernelAbstractions.CPU}, Spectrum{CPU{KernelAbstractions....
├ planet: Earth{Float32}
├ atmosphere: EarthAtmosphere{Float32}
├ coriolis: Coriolis{Vector{Float32}}
├ geopotential: Geopotential{Vector{Float32}}
├ adiabatic_conversion: AdiabaticConversion{Vector{Float32}}
├ particle_advection: Nothing
├ initial_conditions: InitialConditions{ZonalWind{Float32}, PressureOnOrography, JablonowskiTem...
├ forcing: Nothing
├ drag: SpeedLimitDrag{Float32}
├ random_process: Nothing
├ tracers: Dict{Symbol, Tracer}
├ orography: EarthOrography{Float32, Field{Float32, 1, Vector{Float32}, OctahedralGaussianGrid{...
├ land_sea_mask: EarthLandSeaMask{Float32, Field{Float32, 1, Vector{Float32}, OctahedralGaussia...
├ ocean: SlabOcean{Float32}
├ sea_ice: ThermodynamicSeaIce{Flo

In [9]:
add!(model, SpeedyWeather.SnowDepthOutput())
add!(model, SpeedyWeather.SnowMeltOutput())
add!(model, SpeedyWeather.RadiationOutput())
add!(model, SpeedyWeather.OceanOutput())
add!(model, SpeedyWeather.PrecipitationOutput())
add!(model, SpeedyWeather.HumidityOutput())

NetCDFOutput{Field{Float32, 1, Vector{Float32}, FullGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ status: inactive/uninitialized
├ write restart file: true (if active)
├ interpolator: AnvilInterpolator{Float32, RingGrids.GridGeometry{OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Vector{Float32}, Vector{Int64}, Vector{UnitRange{Int64}}}, RingGrids.AnvilLocator{Vector{Float32}, Vector{Int64}}}
├ path: /Users/woodh/Documents/rotation_project/stability/output.nc (overwrite=false)
├ frequency: 21600 seconds
└┐ variables:
 ├ rain_conv: accumulated convective rain [mm]
 ├ sru: Surface shortwave radiation up [W/m^2]
 ├ rain_cond: accumulated large-scale rain [mm]
 ├ lsm: land-sea mask (1=land, 0=sea) [1]
 ├ st: soil temperature [degC]
 ├ tsurf: Surface air temperature [degC]
 ├ temp: temperature [degC]
 ├ srd: Surface shortwave radiation down [W/m^2]
 ├ vor: relative vorticity [s^-1]
 ├ sm: soil moisture [

In [10]:
sim = initialize!(model)

Simulation{PrimitiveWetModel}
├ prognostic_variables::PrognosticVariables{...}
├ diagnostic_variables::DiagnosticVariables{...}
└ model::PrimitiveWetModel{...}

In [11]:
run!(sim, period=Year(5), output=true)